In [70]:
import pandas as pd
import os
from itertools import chain
import glob
import fnmatch
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from enum import Enum
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import json

In [73]:
class LabelType(Enum):
    MARGIN_OF_VICTORY = "mov"
    WIN_PROBABILITY = "win_probability"

class TournamentDivision(Enum):
    MEN = "M"
    WOMEN = "W"

In [ ]:
SEASON = 2026
DIVISION = TournamentDivision.MEN
LABEL_TYPE = LabelType.WIN_PROBABILITY
REFRESH_KENPOM = False
EPOCHS = 100

# Data collection

In [30]:
DATA_DIRS = [f'march-machine-learning-mania-{SEASON}', '../kenpom']

CSV = {}
for path in list(chain(*map(lambda x: glob.glob(x + '/*.csv'), DATA_DIRS))):
    CSV[os.path.basename(path).split('.')[0]] = pd.read_csv(path, encoding='cp1252')
print(CSV.keys())

dict_keys(['MNCAATourneyDetailedResults', 'SampleSubmissionStage2', 'WSecondaryTourneyTeams', 'WNCAATourneySlots', 'MNCAATourneyCompactResults', 'MSeasons', 'SampleSubmissionStage1', 'WTeams', 'MRegularSeasonDetailedResults', 'WNCAATourneyDetailedResults', 'MNCAATourneySlots', 'MGameCities', 'MConferenceTourneyGames', 'WNCAATourneyCompactResults', 'WSecondaryTourneyCompactResults', 'WSeasons', 'Cities', 'WRegularSeasonCompactResults', 'WTeamSpellings', 'WRegularSeasonDetailedResults', 'MRegularSeasonCompactResults', 'WNCAATourneySeeds', 'MNCAATourneySeedRoundSlots', 'WConferenceTourneyGames', 'WTeamConferences', 'MTeamConferences', 'MTeamCoaches', 'MMasseyOrdinals', 'Conferences', 'MTeams', 'WGameCities', 'MNCAATourneySeeds', 'MSecondaryTourneyTeams', 'MTeamSpellings', 'SeedBenchmarkStage1', 'MSecondaryTourneyCompactResults', 'kenpom_2011-2024', 'kenpom_2025-03-19', 'kenpom_2025'])


In [31]:
manual_team_additions = [
    ('Alabama A&M;', 1105),
    ('Arkansas Little Rock', 1114),
    ('Arkansas Pine Bluff', 1115),
    ('Bethune Cookman', 1126),
    ('Cal St. Bakersfield', 1167),
    ('Dixie St.', 1469),
    ('Florida A&M;', 1197),
    ('Houston Christian', 1223),
    ('IUPU Fort Wayne', 1236),
    ('Illinois Chicago', 1227),
    ('LIU', 1254),
    ('Louisiana Lafayette', 1418),
    ('Louisiana Monroe', 1419),
    ('Louisiana St.', 1261),
    ('MD Baltimore County', 1420),
    ('Maryland Eastern Shore', 1271),
    ('Mississippi Valley St.', 1290),
    ('Missouri Kansas City', 1282),
    ('Nevada Las Vegas', 1424),
    ('NJ Inst of Technology', 1312),
    ('North Carolina A&T;', 1299),
    ('Prairie View A&M;', 1341),
    ('Queens', 1474),
    ('Saint Francis', 1384),
    ('Southeast Missouri St.', 1369),
    ('Southeast Missouri', 1369),
    ('St. Francis NY', 1383),
    ('St. Francis PA', 1384),
    ('St. Louis', 1387),
    ('Texas A&M Corpus Chris', 1394),
    ('St. Thomas', 1472),
    ('Tarleton St.', 1470),
    ('Tennessee Martin', 1404),
    ('Texas A&M;', 1401),
    ('Texas A&M; Commerce', 1477),
    ('Texas A&M; Corpus Chris', 1394),
    ('Texas El Paso', 1431),
    ('Texas Pan American', 1410),
    ('UT Rio Grande Valley', 1410),
    ('Virginia Military Inst', 1440),
    ('Wisconsin Green Bay', 1453),
    ('Wisconsin Milwaukee', 1454),
]
lower_manual_team_additions = [(team_name_spelling.lower(), team_id) for (team_name_spelling, team_id) in manual_team_additions]
manual_df = pd.DataFrame([{'TeamNameSpelling': tns, 'TeamID': tid} for (tns, tid) in lower_manual_team_additions])

mteam_spellings = CSV['MTeamSpellings']
mteam_spellings = pd.concat([mteam_spellings, manual_df], ignore_index=True)
# Remove duplicate team spellings while keeping the first occurrence
mteam_spellings = mteam_spellings.drop_duplicates(subset=['TeamNameSpelling'], keep='first')

teams_df = pd.merge(mteam_spellings, CSV['MTeams'], on='TeamID', how='inner').drop(['FirstD1Season', 'LastD1Season'], axis=1)
display(teams_df)

,TeamNameSpelling,TeamID,TeamName
0,a&m-corpus chris,1394,TAM C. Christi
1,a&m-corpus christi,1394,TAM C. Christi
2,abilene chr,1101,Abilene Chr
3,abilene christian,1101,Abilene Chr
4,abilene-christian,1101,Abilene Chr
...,...,...,...
1210,texas pan american,1410,UTRGV
1211,ut rio grande valley,1410,UTRGV
1212,virginia military inst,1440,VMI
1213,wisconsin green bay,1453,WI Green Bay


## Kenpom data
### (only for Men's, n/a for Women's)

In [32]:
# Scrape kenpom for most updated ratings in the current season
def parse_url(
    url,
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
):  
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    return soup


# Headers in the correct order as found in the html table
column_definitions = [
    {"name": "Rk", "type": "int"},
    {"name": "Team", "type": "str"},
    {"name": "Conf", "type": "str"},
    {"name": "W-L", "type": "str"},
    {"name": "NetRtg", "type": "float"},
    {"name": "ORtg", "type": "float"},
    {"name": "seed_ORtg", "type": "int"},
    {"name": "DRtg", "type": "float"},
    {"name": "seed_DRtg", "type": "int"},
    {"name": "AdjT", "type": "float"},
    {"name": "seed_AdjT", "type": "int"},
    {"name": "Luck", "type": "float"},
    {"name": "seed_Luck", "type": "int"},
    {"name": "sos_NetRtg", "type": "float"},
    {"name": "seed_sos_NetRtg", "type": "int"},
    {"name": "sos_ORtg", "type": "float"},
    {"name": "seed_sos_ORtg", "type": "int"},
    {"name": "sos_DRtg", "type": "float"},
    {"name": "seed_sos_DRtg", "type": "int"},
    {"name": "ncsos_NetRtg", "type": "float"},
    {"name": "seed_ncsos_NetRtg", "type": "int"}
]

def extract_kenpom(soup, write_to_csv = False):
    # Find the table with rankings
    table = soup.find("table", {"id": "ratings-table"})
    assert table is not None, "Error: Ratings table not found!"
    
    # Extract team data
    teams = []
    
    for row in table.find("tbody").find_all("tr"):
        cols = row.find_all("td")
    
        # assert len(cols) == len(column_definitions), f"Error: Expected {len(column_definitions)} columns but found {len(cols)} in row: {row}"
    
        if (len(cols) == len(column_definitions)):
            row_data = []
            for i, col in enumerate(cols):
                text_value = col.text.strip()
    
                # Special handling for 'Team' column (avoid capturing seed numbers)
                if column_definitions[i]["name"] == "Team":
                    team_name = col.find("a").text.strip() if col.find("a") else col.text.strip()
                    row_data.append(team_name)
                else:
                    col_type = column_definitions[i]["type"]
                    if col_type == "int":
                        row_data.append(int(text_value))
                    elif col_type == "float":
                        row_data.append(float(text_value))
                    else:
                        row_data.append(text_value)  # Keep as string for other columns
        
            teams.append(row_data)
    
    # Convert to DataFrame
    refreshed_kenpom = pd.DataFrame(teams, columns=[col["name"] for col in column_definitions])

    if (write_to_csv):
        current_date = datetime.today().strftime('%Y-%m-%d')
        refreshed_kenpom.to_csv(f'../kenpom/kenpom_{current_date}.csv', index=False)

    return refreshed_kenpom

In [35]:
KENPOM_URL = "https://kenpom.com/"

if REFRESH_KENPOM:
    soup = parse_url(KENPOM_URL)
    refreshed_kenpom = extract_kenpom(soup, True)
else:
    # Find the most recent kenpom CSV file
    kenpom_files = sorted(
        [path for path in CSV.keys() if path.startswith("kenpom_")], 
        key=lambda x: x.split('_')[-1],  # Sort by the date in the filename
        reverse=True  # Most recent first
    )

    if kenpom_files:
        print(f'Not refreshing kenpom, using {kenpom_files[0]}')
        refreshed_kenpom = CSV[kenpom_files[0]]  # Load the most recent one
    else:
        refreshed_kenpom = None  # Handle case where no files are found


display(refreshed_kenpom)

Not refreshing kenpom, using kenpom_2025-03-19


,Rk,Team,Conf,W-L,NetRtg,ORtg,seed_ORtg,DRtg,seed_DRtg,AdjT,...,Luck,seed_Luck,sos_NetRtg,seed_sos_NetRtg,sos_ORtg,seed_sos_ORtg,sos_DRtg,seed_sos_DRtg,ncsos_NetRtg,seed_ncsos_NetRtg
0,1,Duke,ACC,31-3,38.27,128.0,3,89.7,4,65.7,...,-0.019,229,10.12,55,112.5,47,102.4,63,9.32,21
1,2,Florida,SEC,30-4,36.24,128.6,1,92.4,9,69.6,...,0.007,159,14.53,16,114.5,17,100.0,19,-1.96,231
2,3,Houston,B12,30-4,35.42,123.2,10,87.8,2,61.4,...,0.015,138,13.67,24,113.6,33,99.9,18,2.57,101
3,4,Auburn,SEC,28-5,35.16,128.5,2,93.3,12,67.6,...,0.030,100,19.11,2,117.1,2,98.0,1,10.40,19
4,5,Tennessee,SEC,27-7,31.20,120.3,18,89.1,3,63.6,...,0.016,135,16.02,6,116.7,4,100.6,32,-0.85,201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,Alabama A&M,SWAC,10-22,-23.50,92.0,361,115.5,334,71.0,...,-0.022,246,-9.70,359,101.1,352,110.8,357,1.86,120
360,361,Coppin St.,MEAC,6-24,-24.29,87.3,363,111.6,276,67.7,...,0.033,92,-4.31,279,105.3,244,109.6,302,4.32,64
361,362,Chicago St.,NEC,4-28,-24.38,91.8,362,116.2,341,68.2,...,-0.024,251,-3.30,247,104.3,286,107.6,184,8.49,29
362,363,Arkansas Pine Bluff,SWAC,6-25,-26.08,95.1,349,121.1,359,71.4,...,-0.074,342,-8.64,354,100.9,356,109.6,301,7.05,41


In [37]:
# kenpom_2024.csv holds kenpom data from 2011-2024 - let's load all of that first
CSV['kenpom'] = CSV['kenpom_2011-2024']

# rename refreshed_kenpom columns to match kenpom_2011-2024.csv
refreshed_kenpom = refreshed_kenpom.rename(columns={
    'Team': 'TeamName',
    'ORtg': 'adj_o',
    'DRtg': 'adj_d',
    'AdjT': 'adj_tempo',
    'Luck': 'luck',
    'sos_ORtg': 'sos_adj_o',
    'sos_DRtg': 'sos_adj_d'
})
# Add the season column
refreshed_kenpom['Season'] = SEASON
# Reorder columns to match kenpom_2011-2024
refreshed_kenpom = refreshed_kenpom[['Season', 'TeamName', 'adj_o', 'adj_d', 'adj_tempo', 'luck', 'sos_adj_o', 'sos_adj_d']]
# Append SEASON data to CSV['kenpom']
CSV['kenpom'] = pd.concat([CSV['kenpom'], refreshed_kenpom], ignore_index=True)

display(CSV['kenpom'])

,Unnamed: 0,Season,TeamName,adj_o,adj_d,adj_tempo,luck,sos_adj_o,sos_adj_d
0,0.0,2011,Ohio St.,125.4,88.4,66.0,0.043,107.4,98.3
1,1.0,2011,Duke,118.8,87.2,70.1,0.006,106.0,97.4
2,2.0,2011,Kansas,119.8,88.3,69.6,0.071,106.1,98.7
3,3.0,2011,Texas,114.0,85.3,67.2,-0.055,105.6,97.8
4,4.0,2011,Purdue,116.1,87.2,67.1,-0.004,108.1,97.3
...,...,...,...,...,...,...,...,...,...
4944,NaN,2025,Alabama A&M,92.0,115.5,71.0,-0.022,101.1,110.8
4945,NaN,2025,Coppin St.,87.3,111.6,67.7,0.033,105.3,109.6
4946,NaN,2025,Chicago St.,91.8,116.2,68.2,-0.024,104.3,107.6
4947,NaN,2025,Arkansas Pine Bluff,95.1,121.1,71.4,-0.074,100.9,109.6


In [40]:
kenpom = CSV['kenpom']

kenpom['TeamName_lower'] = kenpom['TeamName'].str.lower()
kenpom = pd.merge(kenpom, teams_df[['TeamNameSpelling', 'TeamID']], left_on='TeamName_lower', right_on='TeamNameSpelling', how='left')

kenpom = kenpom.drop(['Unnamed: 0', 'TeamName_lower'], axis=1)
kenpom['TeamID'] = kenpom['TeamID'].astype('Int64')

null_team_ids_in_kenpom = kenpom['TeamID'].isna()
null_rows = kenpom[null_team_ids_in_kenpom]
if null_rows.size > 0:
    display(null_rows)
    unique_null_team_names = null_rows['TeamName'].unique()
    print(f'Cannot find team ids for following teams: \n{sorted(unique_null_team_names)}')

assert not null_team_ids_in_kenpom.any(), "Null 'TeamID' values found in kenpom data"
assert not kenpom.duplicated(subset=['Season', 'TeamID']).any(), "Duplicate 'Season' and 'TeamID' combinations found!"

In [41]:
seasons_available = kenpom['Season'].unique()
seasons_available

array([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022,
       2023, 2024, 2025])

# Feature engineering

In [42]:
# Home (H) -> 1
# Away (A) -> -1
# Neutral (N) -> 0
# Flipping location can be done by multipling by -1
location_to_number_map = { 'H': 1, 'A': -1, 'N': 0 }

regular_results = CSV['MRegularSeasonCompactResults']
tourney_results = CSV['MNCAATourneyCompactResults']
assert set(regular_results.columns) == set(tourney_results.columns), "regular_results and tourney_results have different columns"
combined_results = pd.concat([regular_results, tourney_results], ignore_index=True)

available_results = combined_results[combined_results['Season'].isin(seasons_available)].copy()
available_results['WLoc_num'] = available_results['WLoc'].map(location_to_number_map)
available_results['mov'] = available_results['WScore'] - available_results['LScore']
available_results['t1_win'] = 1
display(available_results)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WLoc_num,mov,t1_win
113385,2011,7,1228,79,1414,65,H,0,1,14,1
113386,2011,7,1268,105,1370,76,H,0,1,29,1
113387,2011,7,1338,83,1348,75,H,0,1,8,1
113388,2011,7,1400,83,1298,52,H,0,1,31,1
113389,2011,9,1228,84,1405,45,H,0,1,39,1
...,...,...,...,...,...,...,...,...,...,...,...
195443,2024,146,1301,76,1181,64,N,0,0,12,1
195444,2024,146,1345,72,1397,66,N,0,0,6,1
195445,2024,152,1163,86,1104,72,N,0,0,14,1
195446,2024,152,1345,63,1301,50,N,0,0,13,1


In [43]:
# Rename MMasseyOrdinals as filename can be MMasseyOrdinals_something
if 'MMasseyOrdinals' not in CSV:
    CSV['MMasseyOrdinals'] = pd.concat(map(lambda x: CSV[x], fnmatch.filter(CSV.keys(), 'MMasseyOrdinals*')))

massey = CSV['MMasseyOrdinals']
available_massey = massey[massey['Season'].isin(seasons_available)]

# Group by ['Season', 'RankingDayNum', 'TeamID'] and find the average ordinal rank across all systems
real_massey = available_massey.groupby(['Season', 'RankingDayNum', 'TeamID'], as_index=False)['OrdinalRank'].mean()
assert not real_massey.duplicated(subset=['Season', 'RankingDayNum', 'TeamID']).any(), "Duplicate!"
display(real_massey.head())

,Season,RankingDayNum,TeamID,OrdinalRank
0,2011,0,1102,233.0
1,2011,0,1103,109.0
2,2011,0,1104,64.0
3,2011,0,1105,332.0
4,2011,0,1106,278.0


In [44]:
kenpom_columns = ['adj_o', 'adj_d', 'adj_tempo', 'luck', 'sos_adj_o', 'sos_adj_d']
team1_kenpom_columns = ['t1_' + kc for kc in kenpom_columns]
team2_kenpom_columns = ['t2_' + kc for kc in kenpom_columns]

In [46]:
relevant_available_results = available_results[['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WLoc_num', 'mov', 't1_win']]
relevant_kenpom = kenpom[['Season', 'TeamID'] + kenpom_columns]
relevant_massey = real_massey[['Season', 'RankingDayNum', 'TeamID', 'OrdinalRank']]

In [48]:
joined_results = relevant_available_results.copy().rename(columns={'WLoc_num': 't1_loc_num'})

# add team1 (Wteam) kenpom stats
joined_results = pd.merge(joined_results, relevant_kenpom, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']).drop('TeamID', axis=1).rename(columns=dict(zip(kenpom_columns, team1_kenpom_columns)))

# add massey ordinal rank for team1
joined_results = pd.merge(joined_results, relevant_massey, left_on=['Season', 'DayNum', 'WTeamID'], right_on=['Season', 'RankingDayNum', 'TeamID'], how='left').drop(['TeamID', 'RankingDayNum'], axis=1).rename(columns={'OrdinalRank': 't1_OrdinalRank'})

# add team2 (Lteam) kenpom stats
joined_results = pd.merge(joined_results, relevant_kenpom, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']).drop('TeamID', axis=1).rename(columns=dict(zip(kenpom_columns, team2_kenpom_columns)))

# add massey ordinal rank for team2
joined_results = pd.merge(joined_results, relevant_massey, left_on=['Season', 'DayNum', 'LTeamID'], right_on=['Season', 'RankingDayNum', 'TeamID'], how='left').drop(['TeamID', 'RankingDayNum'], axis=1).rename(columns={'OrdinalRank': 't2_OrdinalRank'})

display(joined_results.tail())
print(joined_results.shape)

,Season,DayNum,WTeamID,LTeamID,t1_loc_num,mov,t1_win,t1_adj_o,t1_adj_d,t1_adj_tempo,...,t1_sos_adj_o,t1_sos_adj_d,t1_OrdinalRank,t2_adj_o,t2_adj_d,t2_adj_tempo,t2_luck,t2_sos_adj_o,t2_sos_adj_d,t2_OrdinalRank
75082,2024,146,1301,1181,0,12,1,112.6,101.8,68.7,...,109.6,103.5,NaN,122.4,97.5,67.1,-0.055,110.2,103.3,NaN
75083,2024,146,1345,1397,0,6,1,126.5,96.7,67.9,...,113.6,101.5,NaN,119.2,91.9,70.0,-0.037,113.4,103.0,NaN
75084,2024,152,1163,1104,0,14,1,125.6,96.1,64.6,...,110.3,103.0,NaN,128.2,102.7,72.9,-0.013,114.3,101.8,NaN
75085,2024,152,1345,1301,0,13,1,126.5,96.7,67.9,...,113.6,101.5,NaN,112.6,101.8,68.7,-0.019,109.6,103.5,NaN
75086,2024,154,1163,1345,0,15,1,125.6,96.1,64.6,...,110.3,103.0,NaN,126.5,96.7,67.9,0.046,113.6,101.5,NaN


(75087, 21)


In [49]:
# without massey ordinal ranks because it seems to be missing a lot of values - to add back, add 't1_OrdinalRank' and 't2_OrdinalRank'
data = joined_results[['t1_loc_num'] + team1_kenpom_columns + team2_kenpom_columns + ['mov', 't1_win']]
flipped_data = data.copy()
flipped_data[team1_kenpom_columns] = data[team2_kenpom_columns]
flipped_data[team2_kenpom_columns] = data[team1_kenpom_columns]

flipped_data['t1_loc_num'] = -data['t1_loc_num']
flipped_data['mov'] = -data['mov']
flipped_data['t1_win'] = 1 - data['t1_win']

display(data.head())
display(flipped_data.head())
print(data.shape, flipped_data.shape)

,t1_loc_num,t1_adj_o,t1_adj_d,t1_adj_tempo,t1_luck,t1_sos_adj_o,t1_sos_adj_d,t2_adj_o,t2_adj_d,t2_adj_tempo,t2_luck,t2_sos_adj_o,t2_sos_adj_d,mov,t1_win
0,1,112.7,90.6,66.9,-0.072,109.5,96.8,100.7,106.3,71.1,-0.049,100.2,102.4,14,1
1,1,109.2,91.2,71.6,-0.088,105.5,100.0,91.2,101.1,73.4,0.046,101.2,103.3,29,1
2,1,119.7,90.8,63.3,-0.003,106.4,97.6,102.4,98.2,67.7,0.064,102.9,100.9,8,1
3,1,114.0,85.3,67.2,-0.055,105.6,97.8,90.8,101.4,70.8,-0.038,99.8,104.9,31,1
4,1,112.7,90.6,66.9,-0.072,109.5,96.8,88.4,112.4,64.4,-0.038,100.0,101.1,39,1


,t1_loc_num,t1_adj_o,t1_adj_d,t1_adj_tempo,t1_luck,t1_sos_adj_o,t1_sos_adj_d,t2_adj_o,t2_adj_d,t2_adj_tempo,t2_luck,t2_sos_adj_o,t2_sos_adj_d,mov,t1_win
0,-1,100.7,106.3,71.1,-0.049,100.2,102.4,112.7,90.6,66.9,-0.072,109.5,96.8,-14,0
1,-1,91.2,101.1,73.4,0.046,101.2,103.3,109.2,91.2,71.6,-0.088,105.5,100.0,-29,0
2,-1,102.4,98.2,67.7,0.064,102.9,100.9,119.7,90.8,63.3,-0.003,106.4,97.6,-8,0
3,-1,90.8,101.4,70.8,-0.038,99.8,104.9,114.0,85.3,67.2,-0.055,105.6,97.8,-31,0
4,-1,88.4,112.4,64.4,-0.038,100.0,101.1,112.7,90.6,66.9,-0.072,109.5,96.8,-39,0


(75087, 15) (75087, 15)


In [51]:
augmented_data = pd.concat([data, flipped_data])
display(augmented_data)

,t1_loc_num,t1_adj_o,t1_adj_d,t1_adj_tempo,t1_luck,t1_sos_adj_o,t1_sos_adj_d,t2_adj_o,t2_adj_d,t2_adj_tempo,t2_luck,t2_sos_adj_o,t2_sos_adj_d,mov,t1_win
0,1,112.7,90.6,66.9,-0.072,109.5,96.8,100.7,106.3,71.1,-0.049,100.2,102.4,14,1
1,1,109.2,91.2,71.6,-0.088,105.5,100.0,91.2,101.1,73.4,0.046,101.2,103.3,29,1
2,1,119.7,90.8,63.3,-0.003,106.4,97.6,102.4,98.2,67.7,0.064,102.9,100.9,8,1
3,1,114.0,85.3,67.2,-0.055,105.6,97.8,90.8,101.4,70.8,-0.038,99.8,104.9,31,1
4,1,112.7,90.6,66.9,-0.072,109.5,96.8,88.4,112.4,64.4,-0.038,100.0,101.1,39,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75082,0,122.4,97.5,67.1,-0.055,110.2,103.3,112.6,101.8,68.7,-0.019,109.6,103.5,-12,0
75083,0,119.2,91.9,70.0,-0.037,113.4,103.0,126.5,96.7,67.9,0.046,113.6,101.5,-6,0
75084,0,128.2,102.7,72.9,-0.013,114.3,101.8,125.6,96.1,64.6,0.043,110.3,103.0,-14,0
75085,0,112.6,101.8,68.7,-0.019,109.6,103.5,126.5,96.7,67.9,0.046,113.6,101.5,-13,0


In [52]:
x = augmented_data.drop(['mov', 't1_win'], axis=1)
y_mov = augmented_data['mov']
y_win = augmented_data['t1_win']

# Splitting the data
x_train_mov, x_test_mov, y_train_mov, y_test_mov = train_test_split(x, y_mov, test_size=0.2, random_state=42)
x_train_win, x_test_win, y_train_win, y_test_win = train_test_split(x, y_win, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()

x_train_mov_scaled = scaler.fit_transform(x_train_mov)
x_test_mov_scaled = scaler.transform(x_test_mov)

x_train_win_scaled = scaler.fit_transform(x_train_mov)
x_test_win_scaled = scaler.transform(x_test_mov)

In [53]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the mov model
model_mov = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(x_train_mov_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)  # Output layer for regression should have 1 unit without activation
])

# Compile the mov model
model_mov.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['mean_absolute_error'])

/Users/alan_chen/.pyenv/versions/3.10.14/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [75]:
history_mov = model_mov.fit(x_train_mov_scaled, y_train_mov, validation_split=0.2, epochs=EPOCHS, batch_size=32, verbose=1)

Epoch 1/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 672us/step - loss: 104.9980 - mean_absolute_error: 8.0504 - val_loss: 104.4428 - val_mean_absolute_error: 8.0238
Epoch 2/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 643us/step - loss: 105.4037 - mean_absolute_error: 8.0610 - val_loss: 104.5933 - val_mean_absolute_error: 8.0147
Epoch 3/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 665us/step - loss: 105.6197 - mean_absolute_error: 8.0648 - val_loss: 104.2702 - val_mean_absolute_error: 8.0118
Epoch 4/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 604us/step - loss: 104.6452 - mean_absolute_error: 8.0371 - val_loss: 104.4175 - val_mean_absolute_error: 8.0252
Epoch 5/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 606us/step - loss: 104.9969 - mean_absolute_error: 8.0563 - val_loss: 104.4750 - val_mean_absolute_error: 8.0104
Epoch 6/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 599us/step - loss: 104.5672 - mean_absolute_error: 8.0259 - val_loss: 104.3345 - val_mean_absolute_error: 8.0216
Epoch 7/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 

In [76]:
model_win = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(x_train_win_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Changed for binary classification
])

model_win.compile(optimizer='adam',
              loss='binary_crossentropy',  # Changed for binary classification
              metrics=['accuracy'])  # Common metric for classification

/Users/alan_chen/.pyenv/versions/3.10.14/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [77]:
history_win = model_win.fit(x_train_win_scaled, y_train_win, validation_split=0.2, epochs=EPOCHS, batch_size=32, verbose=1)

Epoch 1/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 699us/step - accuracy: 0.7646 - loss: 0.4835 - val_accuracy: 0.7723 - val_loss: 0.4668
Epoch 2/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 680us/step - accuracy: 0.7732 - loss: 0.4660 - val_accuracy: 0.7738 - val_loss: 0.4671
Epoch 3/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 678us/step - accuracy: 0.7739 - loss: 0.4665 - val_accuracy: 0.7718 - val_loss: 0.4670
Epoch 4/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 664us/step - accuracy: 0.7729 - loss: 0.4660 - val_accuracy: 0.7716 - val_loss: 0.4688
Epoch 5/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 665us/step - accuracy: 0.7747 - loss: 0.4672 - val_accuracy: 0.7736 - val_loss: 0.4663
Epoch 6/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 670us/step - accuracy: 0.7751 - loss: 0.4634 - val_accuracy: 0.7726 - val_loss: 0.4671
Epoch 7/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 677us/step - accuracy: 0.7747 - loss: 0.4627 - val_accuracy: 0.7723 - val_loss: 0.4683
Epoch 8/100
3004/3004 ━━━━━━━━━━━━━━━━━━━━ 2s 672us/step - accuracy: 

In [78]:
# Evaluate the model
test_loss_mov, test_mae = model_mov.evaluate(x_test_mov, y_test_mov, verbose=1)
print('\nTest loss mov:', test_mae)

test_loss_win, test_accuracy = model_win.evaluate(x_test_win, y_test_win, verbose=1)
print(f'Test Loss win: {test_loss_win}, Test Accuracy: {test_accuracy}')


939/939 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - loss: 313.9200 - mean_absolute_error: 14.5351

Test loss mov: 14.50028133392334
939/939 ━━━━━━━━━━━━━━━━━━━━ 0s 408us/step - accuracy: 0.4978 - loss: 9.1980
Test Loss win: 9.17327880859375, Test Accuracy: 0.5004494786262512


In [79]:
def get_team_kenpom_by_id(teamID, season=SEASON):
    return kenpom.loc[(kenpom['Season'] == season) & (kenpom['TeamID'] == teamID)].iloc[0]

def get_team_kenpom_by_team_name(team_name, season=SEASON):
    team = teams_df.loc[teams_df['TeamNameSpelling'] == team_name.lower()]
    assert not team.empty, f"No team found with spelling {team_name}"
    teamID = team.iloc[0]['TeamID']
    
    return get_team_kenpom_by_id(teamID, season)

def get_team_features(team_kenpom):
    return [team_kenpom['adj_o'], team_kenpom['adj_d'], team_kenpom['adj_tempo'], team_kenpom['luck'], team_kenpom['sos_adj_o'], team_kenpom['sos_adj_d']]
#     return [team_kenpom[kenpom_columns]]
    
def is_number(value):
    return isinstance(value, (int, float, complex))

def mov_to_win_prob(mov, k=0.14):
    """
    Convert margin of victory to win probability.
    
    :param mov: Predicted margin of victory (positive for Team 1 win, negative for Team 2 win)
    :param k: Constant determining the steepness of the function
    :return: Probability of Team 1 winning
    """
    win_prob = 1 / (1 + np.exp(-k * mov))
    return win_prob

def predict_matchup(team1, team2, season=SEASON, location=0, label_type=LABEL_TYPE):
    if is_number(team1):
        team1_kenpom = get_team_kenpom_by_id(team1, season)
    else:
        team1_kenpom = get_team_kenpom_by_team_name(team1, season)
    if is_number(team2):
        team2_kenpom = get_team_kenpom_by_id(team2, season)
    else:
        team2_kenpom = get_team_kenpom_by_team_name(team2, season)
        
    
    team1_features = get_team_features(team1_kenpom)
    team2_features = get_team_features(team2_kenpom)
    
    column_names = ['t1_loc_num'] + team1_kenpom_columns + team2_kenpom_columns
    
    full_features = [location] + team1_features + team2_features
    full_features_flipped = [-location] + team2_features + team1_features
    
    full_features_scaled = scaler.transform(pd.DataFrame([full_features], columns=column_names))
    full_features_flipped_scaled = scaler.transform(pd.DataFrame([full_features_flipped], columns=column_names))

    if (label_type == LabelType.MARGIN_OF_VICTORY):
        predicted_mov = model_mov.predict(full_features_scaled, verbose=0)[0][0]
        predicted_mov_flipped = model_mov.predict(full_features_flipped_scaled, verbose=0)[0][0]
        final_predicted_mov = (predicted_mov - predicted_mov_flipped) / 2
        derived_win = mov_to_win_prob(final_predicted_mov)
        
        return derived_win

    elif (label_type == LabelType.WIN_PROBABILITY):
        predicted_win = model_win.predict(full_features_scaled, verbose=0)[0][0]
        predicted_win_flipped = model_win.predict(full_features_flipped_scaled, verbose=0)[0][0]
        final_predicted_win = (predicted_win + (1.0 - predicted_win_flipped)) / 2 

        return final_predicted_win

    return None

In [95]:
print(predict_matchup(1222, 1188, SEASON, 0, LabelType.MARGIN_OF_VICTORY)) # houston vs siue
print(predict_matchup('houston', 'siue', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('duke', 'florida', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('florida', 'duke', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('ole miss', 'unc', SEASON, 0, LabelType.MARGIN_OF_VICTORY))
print(predict_matchup('unc', 'ole miss', SEASON, 0, LabelType.MARGIN_OF_VICTORY))

0.9838537
0.9838537
0.5605852
0.4394148
0.5378832
0.46211678


In [96]:
print(predict_matchup(1222, 1188, SEASON, 0, LabelType.WIN_PROBABILITY)) # houston vs siue
print(predict_matchup('houston', 'siue', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('duke', 'florida', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('florida', 'duke', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('ole miss', 'unc', SEASON, 0, LabelType.WIN_PROBABILITY))
print(predict_matchup('unc', 'ole miss', SEASON, 0, LabelType.WIN_PROBABILITY))

0.9969006
0.9969006
0.5195602
0.48043975
0.5562067
0.4437933


In [82]:
def prepare_data(seeds):
    seed_dict = seeds.set_index('Seed')['TeamID'].to_dict()
    inverted_seed_dict = {value: key for key, value in seed_dict.items()}

    return seed_dict, inverted_seed_dict

# 2025
seeds = CSV['MNCAATourneySeeds']
seeds = seeds[seeds['Season'] == SEASON]

# display(seeds)

seed_dict, inverted_seed_dict = prepare_data(seeds)

print(seed_dict)
print(inverted_seed_dict)

{'W01': 1181, 'W02': 1104, 'W03': 1458, 'W04': 1112, 'W05': 1332, 'W06': 1140, 'W07': 1388, 'W08': 1280, 'W09': 1124, 'W10': 1435, 'W11': 1433, 'W12': 1251, 'W13': 1103, 'W14': 1285, 'W15': 1352, 'W16a': 1110, 'W16b': 1291, 'X01': 1222, 'X02': 1397, 'X03': 1246, 'X04': 1345, 'X05': 1155, 'X06': 1228, 'X07': 1417, 'X08': 1211, 'X09': 1208, 'X10': 1429, 'X11a': 1400, 'X11b': 1462, 'X12': 1270, 'X13': 1219, 'X14': 1407, 'X15': 1459, 'X16': 1188, 'Y01': 1120, 'Y02': 1277, 'Y03': 1235, 'Y04': 1401, 'Y05': 1276, 'Y06': 1279, 'Y07': 1266, 'Y08': 1257, 'Y09': 1166, 'Y10': 1307, 'Y11a': 1314, 'Y11b': 1361, 'Y12': 1471, 'Y13': 1463, 'Y14': 1252, 'Y15': 1136, 'Y16a': 1106, 'Y16b': 1384, 'Z01': 1196, 'Z02': 1385, 'Z03': 1403, 'Z04': 1268, 'Z05': 1272, 'Z06': 1281, 'Z07': 1242, 'Z08': 1163, 'Z09': 1328, 'Z10': 1116, 'Z11': 1179, 'Z12': 1161, 'Z13': 1213, 'Z14': 1423, 'Z15': 1303, 'Z16': 1313}
{1181: 'W01', 1104: 'W02', 1458: 'W03', 1112: 'W04', 1332: 'W05', 1140: 'W06', 1388: 'W07', 1280: 'W08', 11

In [85]:
def build_win_probability_matrix(label_type=LABEL_TYPE):
    win_probability_matrix = {}
    for team_id in inverted_seed_dict.keys():
        win_probability_matrix[team_id] = {}
        for opponent_id in inverted_seed_dict.keys():
            if team_id == opponent_id:
                continue
            if opponent_id in win_probability_matrix and team_id in win_probability_matrix[opponent_id]:
                win_probability_matrix[team_id][opponent_id] = 1.0 - win_probability_matrix[opponent_id][team_id]
            else:
                win_probability_matrix[team_id][opponent_id] = predict_matchup(team_id, opponent_id, season=SEASON, location=0, label_type=LABEL_TYPE)
    return win_probability_matrix

# Convert float32 values to Python float
def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, np.float32):  # Convert np.float32 to float
        return float(obj)
    return obj

def write_to_json(win_probability_matrix, filename):
    serializable_data = convert_to_serializable(win_probability_matrix)

    # Write to JSON file
    with open(filename, "w") as json_file:
        json.dump(serializable_data, json_file)
    
    print(f'{filename} written successfully!')
    

In [86]:
mov_matrix = build_win_probability_matrix(LabelType.MARGIN_OF_VICTORY)
write_to_json(mov_matrix, f'{SEASON}_mov_win_probability_matrix.json')

2025_mov_win_probability_matrix.json written successfully!


In [87]:
win_matrix = build_win_probability_matrix(LabelType.WIN_PROBABILITY)
write_to_json(win_matrix, f'{SEASON}_win_win_probability_matrix.json')

2025_win_win_probability_matrix.json written successfully!


In [88]:
def build_slots(season=SEASON):
    slots = CSV["MNCAATourneySlots"]
    
    slots = slots[slots['Season'] == season]

    # Select early rounds where 'Slot' does NOT start with 'R'
    play_ins = slots[~slots['Slot'].str.startswith('R')]

    # Select remaining slots where 'Slot' starts with 'R'
    actual_slots = slots[slots['Slot'].str.startswith('R')]

    # Concatenate early rounds at the top
    sorted_slots = pd.concat([play_ins, actual_slots], ignore_index=True)

    return sorted_slots

slots = build_slots()
display(slots)

,Season,Slot,StrongSeed,WeakSeed
0,2025,W16,W16a,W16b
1,2025,X11,X11a,X11b
2,2025,Y11,Y11a,Y11b
3,2025,Y16,Y16a,Y16b
4,2025,R1W1,W01,W16
...,...,...,...,...
62,2025,R4Y1,R3Y1,R3Y2
63,2025,R4Z1,R3Z1,R3Z2
64,2025,R5WX,R4W1,R4X1
65,2025,R5YZ,R4Y1,R4Z1


In [89]:
def simulate(round_slots, seeds, inverted_seeds, win_probability_matrix):
    '''
    Simulates each round of the tournament.

    Parameters:
    - round_slots: DataFrame containing information on who is playing in each round.
    - seeds (dict): Dictionary mapping seed values to team IDs.
    - inverted_seeds (dict): Dictionary mapping team IDs to seed values.
    - wins (DF): DF that includes wins prediction per matchup.
    Returns:
    - list: List with winning team IDs for each match.
    - list: List with corresponding slot names for each match.
    '''
    winners = []
    slots = []

    for slot, strong, weak in zip(round_slots.Slot, round_slots.StrongSeed, round_slots.WeakSeed):
        team_1, team_2 = seeds[strong], seeds[weak]

        team_1_prob = win_probability_matrix[team_1][team_2]

        p = np.array([team_1_prob, 1 - team_1_prob])
        p /= p.sum()  # Normalize to ensure sum is exactly 1.0

        # winner = np.random.choice([team_1, team_2], p=[team_1_prob, 1 - team_1_prob])
        winner = np.random.choice([team_1, team_2], p=p)

        # Append the winner and corresponding slot to the lists
        winners.append(winner)
        slots.append(slot)

        seeds[slot] = winner

    return [inverted_seeds[w] for w in winners], slots


def run_simulation(seeds, round_slots, win_probability_matrix, num_brackets):
    '''
    Runs a simulation of bracket tournaments.

    Parameters:
    - seeds (pd.DataFrame): DataFrame containing seed information.
    - round_slots (pd.DataFrame): DataFrame containing information about the tournament rounds.
    - wins (DF): DF that includes wins prediction per matchup.
    - brackets (int): Number of brackets to simulate.
    Returns:
    - pd.DataFrame: DataFrame with simulation results.
    '''
    # Get relevant data for the simulation
    seed_dict, inverted_seed_dict = prepare_data(seeds)
    # Lists to store simulation results
    results = []
    bracket = []
    slots = []

    # Iterate through the specified number of brackets
    for b in tqdm(range(1, num_brackets + 1)):
        # Run single simulation
        r, s = simulate(round_slots, seed_dict, inverted_seed_dict, win_probability_matrix)

        # Update results
        results.extend(r)
        bracket.extend([b] * len(r))
        slots.extend(s)

    # Create final DataFrame
    result_df = pd.DataFrame({'Bracket': bracket, 'Slot': slots, 'Team': results})

    return result_df

In [90]:
num_brackets = 100000

In [91]:
mov_result = run_simulation(seeds, slots, mov_matrix, num_brackets)
mov_result.insert(0, 'Tournament', 'M')

display(mov_result)

100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [01:08<00:00, 1468.35it/s]


,Tournament,Bracket,Slot,Team
0,M,1,W16,W16a
1,M,1,X11,X11b
2,M,1,Y11,Y11a
3,M,1,Y16,Y16a
4,M,1,R1W1,W01
...,...,...,...,...
6699995,M,100000,R4Y1,Y01
6699996,M,100000,R4Z1,Z01
6699997,M,100000,R5WX,W01
6699998,M,100000,R5YZ,Y01


In [92]:
win_result = run_simulation(seeds, slots, win_matrix, num_brackets)
win_result.insert(0, 'Tournament', 'M')

display(win_result)

100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [01:08<00:00, 1459.19it/s]


,Tournament,Bracket,Slot,Team
0,M,1,W16,W16b
1,M,1,X11,X11b
2,M,1,Y11,Y11b
3,M,1,Y16,Y16a
4,M,1,R1W1,W01
...,...,...,...,...
6699995,M,100000,R4Y1,Y01
6699996,M,100000,R4Z1,Z01
6699997,M,100000,R5WX,W01
6699998,M,100000,R5YZ,Z01


In [93]:
def result_to_submission(result, filename):
    submission = result
    submission.reset_index(inplace=True, drop=True)
    submission.index.names = ['RowId']
    submission.to_csv(filename)

In [94]:
result_to_submission(mov_result, f'{SEASON}_mov_my_submission.csv')
result_to_submission(win_result, f'{SEASON}_win_my_submission.csv')